In [1]:
import pandas as pd
import inspect
import polars as pl
from functime.feature_extractors import FeatureExtractor
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tsfresh
from tsfresh import extract_relevant_features, select_features
from tsfresh.feature_extraction import extract_features
from tsfresh.utilities.dataframe_functions import impute
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [2]:
# =========================
# LOAD
# =========================

train = pl.from_pandas(
    pd.read_csv("X_train.txt", sep=r"\s+", header=None)
)

test = pl.from_pandas(
    pd.read_csv("X_test.txt", sep=r"\s+", header=None)
)

y = pd.read_csv("y_train.txt", header=None).squeeze()
y_test = pd.read_csv("y_test.txt", header=None).squeeze()

train = train.with_row_index("id")
test = test.with_row_index("id")

# =========================
# WIDE -> LONG
# =========================

train_long = train.unpivot(
    index="id",
    variable_name="time",
    value_name="value"
)

test_long = test.unpivot(
    index="id",
    variable_name="time",
    value_name="value"
)

# =========================
# SORT
# =========================

train_long = train_long.sort(["id", "time"])
test_long = test_long.sort(["id", "time"])

KeyboardInterrupt: 

In [ ]:
def extract_ts_features(df: pl.DataFrame) -> pl.DataFrame:
    return df.select([
        pl.col("value").ts.absolute_energy().alias("absolute_energy"),
        pl.col("value").ts.binned_entropy(bin_count=10).alias("binned_entropy"),
        pl.col("value").ts.cid_ce().alias("cid_ce"),
        pl.col("value").ts.lempel_ziv_complexity(threshold = 3).alias("lzc"),
        pl.col("value").ts.longest_streak_above_mean().alias("streak_above"),
        pl.col("value").ts.longest_streak_below_mean().alias("streak_below"),
        pl.col("value").ts.number_crossings().alias("crossings"),
        pl.col("value").ts.number_peaks(support = 3).alias("peaks"),
        pl.col("value").ts.root_mean_square().alias("rms"),
        pl.col("value").ts.variation_coefficient().alias("var_coeff"),
    ])

In [ ]:
ts_ = pl.col("value").ts
feature_names = [
    name for name in dir(ts_)
    if not name.startswith("_")
]
print(feature_names)
ts_feature_exprs = []

for name in feature_names:

    try:
        fn = getattr(ts_, name)

        # try calling feature
        expr = fn()

        # ensure it's usable in aggregation
        expr = expr.alias(name)

        ts_feature_exprs.append(expr)

    except Exception:
        # skip non-scalar / param-required / unstable features
        continue

print(ts_feature_exprs)

['absolute_energy', 'absolute_maximum', 'absolute_sum_of_changes', 'autocorrelation', 'benford_correlation', 'binned_entropy', 'c3', 'change_quantiles', 'cid_ce', 'count_above', 'count_above_mean', 'count_below', 'count_below_mean', 'cusum', 'detrend', 'energy_ratios', 'first_location_of_maximum', 'first_location_of_minimum', 'frac_diff', 'harmonic_mean', 'has_duplicate', 'has_duplicate_max', 'has_duplicate_min', 'index_mass_quantile', 'large_standard_deviation', 'last_location_of_maximum', 'last_location_of_minimum', 'lempel_ziv_complexity', 'linear_trend', 'longest_losing_streak', 'longest_streak_above', 'longest_streak_above_mean', 'longest_streak_below', 'longest_streak_below_mean', 'longest_winning_streak', 'max_abs_change', 'mean_abs_change', 'mean_change', 'mean_n_absolute_max', 'mean_second_derivative_central', 'number_crossings', 'number_peaks', 'percent_reoccurring_points', 'percent_reoccurring_values', 'permutation_entropy', 'range_change', 'range_count', 'range_over_mean', 

In [ ]:
def features(df):

    ts_features = (
        df
        .group_by("id")
        .agg([
            pl.col("value").ts.absolute_energy().alias("absolute_energy"),
            pl.col("value").ts.binned_entropy(bin_count=10).alias("binned_entropy"),
            pl.col("value").ts.cid_ce().alias("cid_ce"),
            pl.col("value").ts.lempel_ziv_complexity(threshold=3).alias("lzc"),
            pl.col("value").ts.longest_streak_above_mean().alias("streak_above"),
            pl.col("value").ts.longest_streak_below_mean().alias("streak_below"),
            pl.col("value").ts.number_crossings().alias("crossings"),
            pl.col("value").ts.number_peaks(support=5).alias("peaks"),
            pl.col("value").ts.root_mean_square().alias("rms"),
            pl.col("value").ts.variation_coefficient().alias("var_coeff"),
        ])
    )

    base_features = (
        df
        .group_by("id")
        .agg([
            pl.col("value").mean().alias("mean"),
            pl.col("value").std().alias("std"),
            pl.col("value").min().alias("min"),
            pl.col("value").max().alias("max"),
            pl.col("value").median().alias("median"),
            pl.col("value").skew().alias("skew"),
            pl.col("value").kurtosis().alias("kurtosis"),
            (pl.col("value") ** 2).sum().alias("energy"),
            pl.col("value").abs().mean().alias("mean_abs"),
        ])
    )

    for lag in [1, 2, 5, 10, 20]:
        df = df.with_columns(
            pl.col("value").shift(lag).over("id").alias(f"lag_{lag}")
        )

    lag_features = (
        df
        .group_by("id")
        .agg([
            pl.corr(pl.col("value"), pl.col("lag_1")).alias("autocorr_1"),
            pl.corr(pl.col("value"), pl.col("lag_2")).alias("autocorr_2"),
            pl.corr(pl.col("value"), pl.col("lag_5")).alias("autocorr_5"),
            pl.corr(pl.col("value"), pl.col("lag_10")).alias("autocorr_10"),
            pl.corr(pl.col("value"), pl.col("lag_20")).alias("autocorr_20"),
        ])
    )

    X = (
        base_features
        .join(ts_features, on="id")
        .join(lag_features, on="id")
    )
    return X

X = features(train_long)
X_test = features(test_long)

In [ ]:
# ==========================================
# TO PANDAS
# ==========================================

X_pd = X.drop("id").to_pandas()
X_test_pd = X_test.drop("id").to_pandas()

# ==========================================
# KEEP NUMERIC ONLY
# ==========================================

X_pd = X_pd.select_dtypes(include=["number"])
X_test_pd = X_test_pd[X_pd.columns]

# ==========================================
# CLEAN NAN/INF
# ==========================================

X_pd = X_pd.replace([float("inf"), -float("inf")], 0)
X_test_pd = X_test_pd.replace([float("inf"), -float("inf")], 0)

X_pd = X_pd.fillna(0)
X_test_pd = X_test_pd.fillna(0)

# ==========================================
# FEATURE SELECTION
# ==========================================

from sklearn.feature_selection import SelectFdr, f_classif

selector = SelectFdr(
    score_func=f_classif
)

X_selected = selector.fit_transform(X_pd, y)
X_test_selected = selector.transform(X_test_pd)

In [ ]:
y = (
    pl.read_csv("y_train.txt", has_header=False)
    .to_series()
    .to_numpy()
)

y_test = (
    pl.read_csv("y_test.txt", has_header=False)
    .to_series()
    .to_numpy()
)

In [ ]:
# ==========================================
# MODEL
# ==========================================

model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3
)

model.fit(X_selected, y)

# ==========================================
# PREDICT
# ==========================================

preds = model.predict(X_test_selected)

# ==========================================
# EVALUATE
# ==========================================

print(
    "Accuracy:",
    accuracy_score(y_test, preds)
)

Accuracy: 0.7210722768917543
